# Get Nadex Contract Locations

Downloads daily settlement PDFs from Nadex's public S3 bucket, extracts US 500 binary
contract strikes, and identifies the strikes that bracket each day's 09:30 market open.

**Output:** `contract_locations.csv` â€” columns: `date`, `above`, `below`

**Run before `barsToCleaning.ipynb`** whenever new trading dates have been added to `GoodOldGoodOld.csv`.

> PDFs are settlement results, available after market close.  
> Same-day (live) contract lookup is not supported by this script.

In [1]:
# â”€â”€ CONFIG â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Filters to isolate US 500 daily binary options expiring at 4:15 PM ET
# Note: only PRODUCT_FILTER and EXPIRY_FILTER are used in text scanning.
# The 4:15PM expiry is unique to Daily contracts â€” no PERIOD_FILTER needed.
PRODUCT_FILTER = "US 500"
EXPIRY_FILTER  = "4:15PM"

PDF_URL     = "https://s3.amazonaws.com/market-data-prod.nadex.com/{date}_tradingResults.pdf"
OUTPUT_FILE = "contract_locations.csv"
BARS_FILE   = "GoodOldGoodOld.csv"
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

In [2]:
import json
import pathlib
import subprocess
import sys

import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed


In [3]:
# Load 1-minute bars and extract the 09:30:00 open price for each trading day
bars = pd.read_csv(BARS_FILE)
# utc=True handles mixed-timezone strings in the CSV; convert to Eastern then strip
# timezone so dates are naive and match what contract_locations.csv stores.
bars['date_only'] = (
    pd.to_datetime(bars['date'], utc=True)
    .dt.tz_convert('US/Eastern')
    .dt.normalize()
    .dt.tz_localize(None)
)

opens = (
    bars[bars['time'] == '09:30:00']
    .groupby('date_only')['open']
    .first()
    .reset_index()
    .rename(columns={'date_only': 'date', 'open': 'market_open'})
)

print(f"Trading days in {BARS_FILE}: {len(opens)}")
print(opens.tail())

Trading days in GoodOldGoodOld.csv: 193
          date  market_open
188 2024-12-09      6096.25
189 2024-12-10      6072.00
190 2024-12-11      6075.75
191 2024-12-12      6082.50
192 2024-12-13      6077.50


In [4]:
# Load existing contract_locations.csv for incremental updates.
# Dates already covered are skipped â€” no redundant PDF downloads.
try:
    existing = pd.read_csv(OUTPUT_FILE, parse_dates=['date'])
    existing['date'] = existing['date'].dt.normalize()
    covered_dates = set(existing['date'])
    print(f"Existing {OUTPUT_FILE} covers {len(covered_dates)} date(s)")
except FileNotFoundError:
    existing = pd.DataFrame(columns=['date', 'above', 'below'])
    covered_dates = set()
    print(f"No existing {OUTPUT_FILE} â€” building from scratch")

dates_to_fetch = [d for d in opens['date'] if d not in covered_dates]
print(f"Dates to fetch: {len(dates_to_fetch)}")

No existing contract_locations.csv â€” building from scratch
Dates to fetch: 193


In [ ]:
_PARSE_PDF_SCRIPT = str(pathlib.Path.cwd() / "parse_pdf.py")


def get_contracts_for_date(date, market_open):
    """Download and parse the Nadex results PDF for `date` via a subprocess.

    Each PDF is parsed in an isolated child process so a hung pdfplumber call
    cannot freeze the worker pool.  Returns (above_strike, below_strike), or
    (None, None) if the PDF is unavailable (weekend / holiday / future date).
    """
    url = PDF_URL.format(date=date.strftime('%Y%m%d'))
    proc = subprocess.Popen(
        [sys.executable, _PARSE_PDF_SCRIPT, url, str(market_open)],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
    )
    try:
        stdout, stderr = proc.communicate(timeout=50)
    except subprocess.TimeoutExpired:
        proc.kill()
        _, stderr = proc.communicate()
        last = stderr.strip().splitlines()[-1] if stderr.strip() else 'no output'
        print(f'  {date.date()} WARNING: subprocess timed out. Last: {last}')
        return None, None
    except Exception as e:
        print(f'  {date.date()} ERROR: {e}')
        return None, None

    if not stdout.strip():
        return None, None

    try:
        data = json.loads(stdout.strip())
    except json.JSONDecodeError:
        return None, None

    if data.get("miss"):
        return None, None

    if data.get("error"):
        print(f'  {date.date()} network error: {data["error"]}')
        return None, None

    above = data.get("above")
    below = data.get("below")
    if above is None:
        print(f'  {date.date()} WARNING: market open {market_open} is above all strikes')
    if below is None:
        print(f'  {date.date()} WARNING: market open {market_open} is below all strikes')
    return above, below


In [ ]:
SAVE_EVERY = 10  # flush to disk every N successful results

market_opens = dict(zip(opens['date'], opens['market_open']))

def _fetch(date):
    above, below = get_contracts_for_date(date, market_opens[date])
    return above, below

def _flush(combined, new_rows, *, final=False):
    """Append new_rows to combined and save. Dedup+sort only on final flush."""
    if not new_rows:
        return combined
    updated = pd.concat([combined, pd.DataFrame(new_rows)], ignore_index=True)
    if final:
        updated = updated.drop_duplicates('date').sort_values('date').reset_index(drop=True)
    updated.to_csv(OUTPUT_FILE, index=False)
    return updated

print(f"Fetching {len(dates_to_fetch)} date(s) with 12 parallel workers (saving every {SAVE_EVERY} results)...")

pending_rows = []

with ThreadPoolExecutor(max_workers=12) as executor:
    futures = {executor.submit(_fetch, d): d for d in dates_to_fetch}
    for i, future in enumerate(as_completed(futures), 1):
        date = futures[future]
        try:
            above, below = future.result(timeout=90)
        except TimeoutError:
            print(f"  {date.date()} TIMEOUT — skipping (pdfplumber likely hung)")
            continue
        except Exception as e:
            print(f"  {date.date()} ERROR: {e}")
            continue

        if above is not None or below is not None:
            pending_rows.append({'date': date, 'above': above, 'below': below})
            print(f"{date.date()}  open={market_opens[date]:.2f}  below={below}  above={above}")
        else:
            print(f"{date.date()}  — no data (holiday / weekend / future date)")

        if i % SAVE_EVERY == 0:
            existing = _flush(existing, pending_rows)
            pending_rows = []
            print(f"  [{i}/{len(dates_to_fetch)}] checkpoint saved ({len(existing)} total rows)")

existing = _flush(existing, pending_rows, final=True)

# Add market_open to every row using the opens map derived from GoodOldGoodOld.csv.
# Covers all dates present in the current bars file; dates from prior sessions
# that are no longer in bars will get NaN (rare edge case).
mo_map = opens.set_index('date')['market_open']
existing['market_open'] = existing['date'].map(mo_map)
col_order = ['date', 'above', 'below', 'market_open']
existing = existing[col_order]
existing.to_csv(OUTPUT_FILE, index=False)
print(f"
Done. Saved {len(existing)} date(s) to {OUTPUT_FILE}")

In [ ]:
# ── FLAG INCOMPLETE ROWS ──────────────────────────────────────────────────────
# Reloads the saved CSV and flags any row missing above, below, or market_open.
# For each flagged row, a likely reason is printed to help with manual lookup.

final = pd.read_csv(OUTPUT_FILE, parse_dates=["date"])
final["date"] = final["date"].dt.normalize()

missing = final[
    final["above"].isna() | final["below"].isna() | final["market_open"].isna()
]

if missing.empty:
    print("All rows have above, below, and market_open — no manual lookup needed.")
else:
    print(f"{len(missing)} row(s) need manual lookup:
")
    for _, row in missing.iterrows():
        date_str = row["date"].strftime("%Y-%m-%d")
        mo = row["market_open"]
        above = row["above"]
        below = row["below"]

        reasons = []

        if pd.isna(mo):
            reasons.append(
                "market_open missing — date not found in GoodOldGoodOld.csv. "
                "Check that barsToCleaning has been run and the bars file is up to date."
            )

        if pd.isna(above) and pd.isna(below):
            reasons.append(
                "above AND below both missing — no Binary Daily US 500 strikes were "
                "found in the PDF for this date. Possible causes: PDF was not published "
                "(holiday / market closure), PDF structure changed, or the download failed."
            )
        elif pd.isna(above) and not pd.isna(mo):
            reasons.append(
                f"above missing — market opened at {mo:.2f}, which was ABOVE every "
                "available Binary Daily strike. No above-strike contract existed that day. "
                "Manual lookup: check nadex.com or the PDF to confirm the highest strike listed."
            )
        elif pd.isna(below) and not pd.isna(mo):
            reasons.append(
                f"below missing — market opened at {mo:.2f}, which was BELOW every "
                "available Binary Daily strike. No below-strike contract existed that day. "
                "Manual lookup: check nadex.com or the PDF to confirm the lowest strike listed."
            )

        print(f"Date : {date_str}")
        print(f"  above       : {above}")
        print(f"  below       : {below}")
        print(f"  market_open : {mo}")
        for r in reasons:
            print(f"  REASON      : {r}")
        nadex_date = row["date"].strftime("%Y%m%d")
        print(f"  PDF URL     : https://s3.amazonaws.com/market-data-prod.nadex.com/{nadex_date}_tradingResults.pdf")
        print()